# 03 - Candidate strategy

## Objetivo

Definir o universo de modelagem supervisionada para o sistema de recomendação de próximo carrinho.

Este notebook cobre:

- Separação dos conjuntos `prior` e `train` a partir do dataset unificado.
- Definição de usuários elegíveis.
- Diagnóstico da estratégia inicial de candidatos.
- Construção da estratégia de candidatos v2 com segmentação por volume histórico.
- Geração de candidatos por recompra, similaridade Jaccard, popularidade por categoria e popularidade global.
- Construção do target supervisionado usando exclusivamente o pedido `train`.
- Diagnóstico inicial do target e do recall ceiling dos candidatos.

## Inputs

- `data/processed/orders_product_unified.parquet`

## Outputs esperados

- `data/features/candidates_v2_with_target.parquet` — pares `user_id-product_id` candidatos com `target`

## Limites deste notebook

Este notebook não calcula features finais de modelagem.

Este notebook não consolida o dataset final de treino.

Este notebook não treina modelos.

Este notebook não implementa baselines.

Este notebook não define a arquitetura MLP.

O conjunto `prior` é usado como histórico para geração de candidatos.

O conjunto `train` é usado exclusivamente para construção e diagnóstico do target supervisionado. Nenhuma informação de `train` é usada como feature.

O conjunto `test` não está disponível no dataset unificado e não é usado neste notebook.

A estratégia de candidatos adotada é a v2, com segmentação por volume histórico e combinação de recompra, similaridade Jaccard, categorias e popularidade global.

---

## 1. Setup inicial

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

pd.set_option("display.max_columns", None)

In [ ]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TEST_SAMPLE_DIR = DATA_DIR / "test_sample"
FEATURES_DIR = DATA_DIR / "features"
CACHED_DIR = DATA_DIR / 'cache'

UNIFIED_DATASET_PATH = PROCESSED_DIR / "orders_product_unified.parquet"
SAMPLE_DATASET_PATH = TEST_SAMPLE_DIR / "orders_product_sample.parquet"

JACCARD_CACHE_PATH = CACHED_DIR / 'jaccard_neighbors_cache.parquet'

OUTPUT_CANDIDATES_PATH = FEATURES_DIR / "candidates_v2_with_target.parquet"

assert UNIFIED_DATASET_PATH.exists(), (
    f"Dataset unificado não encontrado em: {UNIFIED_DATASET_PATH}. "
    "Execute o notebook 01 antes de continuar."
)

print("Arquivos de entrada encontrados com sucesso.")
print(f"Output será salvo em: {FEATURES_DIR}")

Arquivos de entrada encontrados com sucesso.
Output será salvo em: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features


### 1.1 Carregamento dos dados

In [3]:
USE_SAMPLE = False

dataset_path = SAMPLE_DATASET_PATH if USE_SAMPLE else UNIFIED_DATASET_PATH

df = pd.read_parquet(dataset_path)

print(f"Dataset carregado: {dataset_path.name}")
print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")

if USE_SAMPLE:
    print(
        "\n⚠️  ATENÇÃO: amostra auxiliar carregada. "
        "Não use este resultado para conclusões finais."
    )

Dataset carregado: orders_product_unified.parquet
Shape: 33,819,106 linhas x 15 colunas


In [4]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 15 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   product_id              int64  
 2   add_to_cart_order       int64  
 3   reordered               int64  
 4   user_id                 int64  
 5   eval_set                object 
 6   order_number            int64  
 7   order_dow               int64  
 8   order_hour_of_day       int64  
 9   days_since_prior_order  float64
 10  product_name            object 
 11  aisle_id                int64  
 12  department_id           int64  
 13  aisle                   object 
 14  department              object 
dtypes: float64(1), int64(10), object(4)
memory usage: 10.6 GB


In [5]:
df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


---

## 2. Definição do universo de modelagem

Esta seção estabelece o contrato do dataset supervisionado antes de qualquer feature ser calculada.

Todas as decisões tomadas aqui definem o escopo do problema: quais usuários existem, quais pares `user_id-product_id` são candidatos, e o que significa `target = 1` ou `target = 0`.

Erros nesta seção se propagam silenciosamente para todas as features e para o modelo. Por isso, cada etapa é acompanhada de validação explícita.

### 2.1 Separação prior / train

O dataset unificado contém pedidos dos conjuntos `prior` e `train` identificados pela coluna `eval_set`.

- `prior`: histórico de compras do usuário. Será usado exclusivamente para calcular features.
- `train`: próximo pedido conhecido do usuário. Será usado exclusivamente para construir o target supervisionado.

Nenhuma informação de `train` deve entrar no cálculo de features.

In [6]:
df_prior = df[df["eval_set"] == "prior"].copy()
df_train = df[df["eval_set"] == "train"].copy()

assert not df_prior.empty, "df_prior está vazio. Verifique o dataset unificado."
assert not df_train.empty, "df_train está vazio. Verifique o dataset unificado."
assert set(df_prior["eval_set"].unique()) == {"prior"}, "df_prior contém registros fora do conjunto prior."
assert set(df_train["eval_set"].unique()) == {"train"}, "df_train contém registros fora do conjunto train."

print(f"df_prior: {df_prior.shape[0]:,} linhas | {df_prior['user_id'].nunique():,} usuários | {df_prior['order_id'].nunique():,} pedidos")
print(f"df_train: {df_train.shape[0]:,} linhas | {df_train['user_id'].nunique():,} usuários | {df_train['order_id'].nunique():,} pedidos")

df_prior: 32,434,489 linhas | 206,209 usuários | 3,214,874 pedidos
df_train: 1,384,617 linhas | 131,209 usuários | 131,209 pedidos


### 2.2 Usuários elegíveis

Usuários elegíveis são aqueles com exatamente um pedido `train` conhecido.

O notebook 01 já validou que cada usuário possui no máximo um pedido `train`. Aqui essa condição é reafirmada e o conjunto de usuários elegíveis é materializado para uso nas etapas seguintes.

Usuários com `eval_set = test` não possuem produtos conhecidos no próximo pedido e não fazem parte do dataset de modelagem supervisionado.

In [7]:
train_orders_per_user = (
    df_train.groupby('user_id')['order_id']
    .nunique()
)

assert (train_orders_per_user == 1).all(), (
    "Existem usuários com mais de um pedido train. Verifique o dataset unificado."
)

eligible_users = set(df_train['user_id'].unique())

print(f"Usuários elegíveis (com pedidos train): {len(eligible_users):,}")
print(f"Usuário apenas com prior (sem pedido train): {df_prior['user_id'].nunique() - len(eligible_users):,}")
print(f"Porcentagem de usuário elegíveis: {100 * len(eligible_users) / df_prior['user_id'].nunique()}")

Usuários elegíveis (com pedidos train): 131,209
Usuário apenas com prior (sem pedido train): 75,000
Porcentagem de usuário elegíveis: 63.629133548972156


### 2.3 Estratégia de candidatos — versão 2 com segmentação por usuário

**Decisão de design**: A estratégia v1 (apenas recompra) foi descartada porque o teto teórico de 59.86% está abaixo do requisito de negócio (70-75% de recall).

Em vez disso, adotamos **v2 com segmentação por tipo de usuário**, que combina:
- Produtos já comprados (recompra)
- Produtos populares globalmente
- Produtos populares por categoria preferida
- (Opcionalmente) Similaridade usuário-usuário para usuários com histórico suficiente

#### **Segmentação por volume de histórico:**
 
**Usuários P0-P50 (≤ 48 produtos únicos):**
* Recompra + populares globais + populares por categoria
* Candidatos totais: cap em ~150

**Usuários P50-P90 (de 49-179 produtos únicos):**
* Recompra + populares por categoria + recomendações por similaridade
* Candidatos totais: cap em ~200

**Usuários P90+ (179+ produtos únicos):**
* Recompra + similaridade
* Candidatos totais: cap em ~150 (evitar explosão de candidatos)

Essa estratégia garante:
- Cobertura suficiente para atingir 70-75% de recall
- Performance previsível em produção (latência consistente)
- Uso eficiente de embeddings (similaridade ajuda MLP a generalizar)

### 2.3.1 Correlação entre volume de pedidos e produtos únicos no prior

In [8]:
user_prior_stats = (
    df_prior[df_prior['user_id'].isin(eligible_users)]
    .groupby('user_id')
    .agg(
        user_prior_order_count = ('order_id', 'nunique'),
        user_unique_products = ('product_id', 'nunique'),
    )
    .reset_index()
)

corr_spearman = user_prior_stats['user_prior_order_count'].corr(
    user_prior_stats['user_unique_products'], method = 'spearman'
)

pcts = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

print(f"Correlação Spearman (order_count vs unique_products): {corr_spearman:.4f}")
print(f"Total de usuários elegíveis analisados: {len(user_prior_stats):,}")

print("\nDistribuição de user_prior_order_count:")
print(user_prior_stats["user_prior_order_count"].describe(percentiles=pcts))

Correlação Spearman (order_count vs unique_products): 0.6759
Total de usuários elegíveis analisados: 131,209

Distribuição de user_prior_order_count:
count    131209.000000
mean         15.603937
std          16.661077
min           3.000000
10%           3.000000
25%           5.000000
50%           9.000000
75%          19.000000
90%          37.000000
95%          51.000000
99%          87.000000
max          99.000000
Name: user_prior_order_count, dtype: float64


In [9]:
print("\nDistribuição de user_unique_products:")
print(user_prior_stats["user_unique_products"].describe(percentiles=pcts))


Distribuição de user_unique_products:
count    131209.000000
mean         64.589022
std          56.564099
min           1.000000
10%          13.000000
25%          25.000000
50%          48.000000
75%          86.000000
90%         139.000000
95%         179.000000
99%         266.000000
max         726.000000
Name: user_unique_products, dtype: float64


**Resultados**

Correlação Spearman de **0.6759** entre número de pedidos e produtos únicos no prior — moderada-forte, confirmando que usuários com mais pedidos tendem a acumular mais produtos únicos. 

Spearman foi escolhido por ser robusto a distribuições assimétricas e a outliers, como é o caso de `user_unique_products` (mediana 48, max 726).

Os percentis confirmam os limiares de segmentação adotados:

| Grupo | Limiar de `user_unique_products` | Usuários (~) |
|-------|----------------------------------|--------------|
| P0-P50 | ≤ 48 produtos únicos | 65,600 |
| P50-P90 | 49 – 139 produtos únicos | 52,500 |
| P90+ | > 139 produtos únicos | 13,100 |

### 2.3.2 Análise de cobertura por cap de recompra

In [10]:
P50_THRESHOLD = 48
P90_THRESHOLD = 139

user_product_freq = (
    df_prior[df_prior['user_id'].isin(eligible_users)]
    .groupby(['user_id', 'product_id'])['order_id']
    .nunique()
    .reset_index(name = 'buy_count')
)

user_product_freq['rank'] = (
    user_product_freq.groupby('user_id')['buy_count']
    .rank(method = 'first', ascending = False)
    .astype(int)
)

user_group = user_prior_stats[['user_id', 'user_unique_products']].copy()

user_group['group'] = pd.cut(
    user_group['user_unique_products'],
    bins = [0, P50_THRESHOLD, P90_THRESHOLD, float('inf')],
    labels = ['P0-P50', 'P50-P90', "P90+"], 
)

print(f"user_product_freq: {len(user_product_freq):,} pares (user, product)")
print(f"\nDistribuição por grupo:")
print(user_group["group"].value_counts().sort_index().to_string())

user_product_freq: 8,474,661 pares (user, product)

Distribuição por grupo:
group
P0-P50     66805
P50-P90    51332
P90+       13072


In [11]:
CAPS = [50, 100, 150, 175, 200, 300]

train_products = (
    df_train[df_train['user_id'].isin(eligible_users)][['user_id', 'product_id']]
    .drop_duplicates()
    .assign(target = 1)
)

train_set = train_products[["user_id", "product_id"]].copy()
train_with_group = train_set.merge(user_group[["user_id", "group"]], on="user_id")

results = []

for cap in CAPS:
    cands = user_product_freq[user_product_freq["rank"] <= cap][["user_id", "product_id"]]
    covered = cands.merge(train_set, on=["user_id", "product_id"], how="inner")
    covered_with_group = covered.merge(user_group[["user_id", "group"]], on="user_id")

    for grp in ["P0-P50", "P50-P90", "P90+"]:
        total = (train_with_group["group"] == grp).sum()
        hits = (covered_with_group["group"] == grp).sum()
        results.append({
            "cap": cap,
            "group": grp,
            "recall_ceiling": hits / total if total > 0 else 0,
        })

coverage_df = pd.DataFrame(results)

pivot_recall = (
    coverage_df
    .pivot(index="cap", columns="group", values="recall_ceiling")
    .rename_axis(None, axis=1)
    .mul(100)
    .round(2)
)

print("Recall ceiling por cap e grupo (%):\n")
print(pivot_recall.to_string())

Recall ceiling por cap e grupo (%):

     P0-P50  P50-P90   P90+
cap                        
50    49.38    52.43  49.25
100   49.38    61.91  62.88
150   49.38    62.98  70.43
175   49.38    62.98  72.54
200   49.38    62.98  73.71
300   49.38    62.98  75.13


**Resultados da análise de cobertura**

A análise de recompra ranqueada por frequência de compra revela padrões distintos por grupo:

- **P0-P50**: Recall achata em 49.38% a partir de cap=50. O pool de recompra é pequeno; usuários deste grupo não geram truncamento significativo mesmo com caps altos.
- **P50-P90**: Recall sobe de 52.43% (cap=50) para 62.98% (cap=150), depois estabiliza. A curva achata claramente entre 100-150.
- **P90+**: Recall cresce até cap=300 (49.25% → 75.13%), mas com desaceleração perceptível após cap=150 (70.43% → 72.54%, apenas +2.11 pp).

**Caps finais de recompra por grupo: justificados pelos dados.**

#### **Alocação final de candidatos por grupo**

| Grupo | Recompra | Globais | Categoria | Similaridade | **Total** | Overflow → |
|-------|----------|---------|-----------|--------------|----------|-----------|
| **P0-P50** | 50 | 50 | 50 | 200 | **350** | Sim→Cat→Global |
| **P50-P90** | 125 | — | 25 | 200 | **350** | Sim→Cat→Global|
| **P90+** | 175 | — | — | 150 | **350** | Sim→Cat→Global |

**Lógica de alocação:**
1. Recompra ocupa o slot prioritário (maior recall no grupo)
2. Descoberta (globais + categoria) preenche o restante para P0-P50 e P50-P90
3. Similaridade complementa todos os grupos (vizinhos Jaccard)
4. Overflows (candidatos de recompra não preenchidos) são preenchidos exclusivamente por similaridade, garantindo cobertura balanceada

**Deduplicação obrigatória** entre todas as fontes — cada par (user_id, product_id) aparece no máximo uma vez no set final.

### 2.3.3 Popularidade global

In [12]:
GLOBAL_POOL_SIZE = 250

global_popularity = (
    df_prior
    .groupby('product_id')
    .size()
    .reset_index(name = 'global_buy_count')
    .sort_values('global_buy_count', ascending = False)
    .head(GLOBAL_POOL_SIZE)
    .reset_index(drop = True)
    .assign(global_rank = lambda d: range(1, len(d) + 1))
)

assert len(global_popularity) == GLOBAL_POOL_SIZE, (
    f"Pool global tem {len(global_popularity)} produtos, esperado {GLOBAL_POOL_SIZE}."
)
assert global_popularity['product_id'].is_unique, (
    "Produtos duplicados no pool global."
)

print(f"Pool global: {len(global_popularity):,} produtos")
print(f"Rank 1   — compras: {global_popularity['global_buy_count'].iloc[0]:,}")
print(f"Rank {GLOBAL_POOL_SIZE} — compras: {global_popularity['global_buy_count'].iloc[-1]:,}")
global_popularity.head()

Pool global: 250 produtos
Rank 1   — compras: 472,565
Rank 250 — compras: 16,246


,product_id,global_buy_count,global_rank
0,24852,472565,1
1,13176,379450,2
2,21137,264683,3
3,21903,241921,4
4,47209,213584,5


### 2.3.4 Popularidade por categoria

In [13]:
user_aisle_freq = (
    df_prior[df_prior['user_id'].isin(eligible_users)]
    .groupby(['user_id', 'aisle_id'])
    .size()
    .reset_index(name = 'user_aisle_count')
)

user_aisle_freq['user_aisle_weight'] = (
    user_aisle_freq['user_aisle_count'] /
    user_aisle_freq.groupby('user_id')['user_aisle_count'].transform('sum')
)

print(f"Pares (user, aisle): {len(user_aisle_freq):,}")
print(f"Média de aisles distintos por usuário: {user_aisle_freq.groupby('user_id').size().mean():.1f}")
user_aisle_freq.head()

Pares (user, aisle): 3,647,699
Média de aisles distintos por usuário: 27.8


,user_id,aisle_id,user_aisle_count,user_aisle_weight
0,1,21,8,0.135593
1,1,23,12,0.203390
2,1,24,5,0.084746
3,1,45,1,0.016949
4,1,53,2,0.033898


In [14]:
aisle_product_popularity = (
    df_prior
    .groupby(['aisle_id', 'product_id'])
    .size()
    .reset_index(name = 'aisle_product_count')
)

aisle_product_popularity['aisle_product_rank'] = (
    aisle_product_popularity
    .groupby('aisle_id')['aisle_product_count']
    .rank(method = 'first', ascending = False)
    .astype(int)
)

print(f"Pares (aisle, product): {len(aisle_product_popularity):,}")
print(f"Aisles cobertos: {aisle_product_popularity['aisle_id'].nunique():,}")
print(f"Média de produtos por aisle: {aisle_product_popularity.groupby('aisle_id').size().mean():.1f}")

aisle_product_popularity.head()

Pares (aisle, product): 49,677
Aisles cobertos: 134
Média de produtos por aisle: 370.7


,aisle_id,product_id,aisle_product_count,aisle_product_rank
0,1,209,80,82
1,1,554,1284,13
2,1,886,24,117
3,1,1600,204,62
4,1,2539,695,29


### 2.3.5 Similaridade Jaccard usuário-usuário

**Por que a Similaridade de Jaccard?**

Trabalhamos com **feedback implícito** (dados binários: comprou [1] ou não comprou [0]). O Jaccard é a escolha ideal para este cenário por três motivos:

* **Ignora Co-ausências (Zeros):** Se dois usuários não compraram os mesmos 40.000 produtos do site, isso não significa que eles são parecidos. Diferente da *Distância Euclidiana*, o Jaccard ignora os zeros compartilhados.
* **Focado em Dados Binários:** Métodos como a *Correlação de Pearson* dependem de notas (ex: 1 a 5 estrelas) e perdem o sentido matemático em matrizes binárias.
* **Penaliza Desproporção:** Ao contrário da *Similaridade de Cosseno*, o Jaccard penaliza severamente a comparação entre um usuário comum (comprou 3 itens) e um "super-comprador/revendedor" (comprou 1.000 itens), evitando recomendações distorcidas.

**A Fórmula**
A similaridade entre o conjunto de produtos do Usuário A ($S_A$) e do Usuário B ($S_B$) é calculada por:

$$J(A,B) = \frac{|S_A \cap S_B|}{|S_A \cup S_B|}$$

Onde:
* **$|S_A \cap S_B|$ (Interseção):** Quantidade de produtos comprados por ambos.
* **$|S_A \cup S_B|$ (União):** Total de produtos únicos somando os dois carrinhos ($|S_A| + |S_B| - \text{Interseção}$).

---

**Estratégia de Implementação e Otimização**

Comparar todos os 131k usuários entre si geraria um custo computacional proibitivo de $O(n^2)$ (~17 bilhões de combinações). Para viabilizar o cálculo em minutos, o script adota três estratégias de engenharia de dados:

**A. Matriz Esparsa Binária ($M$)**
Os dados são convertidos em uma matriz onde as linhas são usuários e as colunas são produtos. Usando representação esparsa (`scipy.sparse`), o sistema armazena apenas as posições preenchidas com `1` (compras feitas), reduzindo drasticamente o uso de memória RAM.

**B. O Truque da Transposta ($M \times M^T$)**
A multiplicação da matriz binária por sua própria transposta calcula, via álgebra linear vetorizada, a **interseção de todos os pares de usuários simultaneamente**. 

**C. Processamento em Lotes (Batches)**
Mesmo com matrizes esparsas, instanciar a matriz de similaridade completa quebraria o limite de memória. O pipeline divide o processamento em blocos (`BATCH_SIZE = 500`), realizando a união e a divisão do Jaccard em lotes controlados.

**D. Filtros de Poda (*Pruning*)**
* **`MIN_SHARED`:** Descarta imediatamente pares com baixa intersecção (evita computar Jaccard irrelevante).
* **`N_NEIGHBORS`:** Mantém armazenado no dicionário final apenas os vizinhos mais próximos de cada usuário, otimizando o pipeline de extração de produtos.

In [15]:
eligible_list = sorted(eligible_users)
user_to_idx = {u: i for i, u in enumerate(eligible_list)}

df_prior_dedup = (
    df_prior[df_prior['user_id'].isin(eligible_users)][['user_id', 'product_id']]
    .drop_duplicates()
)

all_products = sorted(df_prior_dedup['product_id'].unique())
product_to_idx = {p: i for i, p in enumerate(all_products)}

rows = df_prior_dedup['user_id'].map(user_to_idx).values
cols = df_prior_dedup['product_id'].map(product_to_idx).values

n_users = len(eligible_list)
n_products = len(all_products)

matrix = csr_matrix(
    (np.ones(len(rows), dtype = np.float32), (rows, cols)),
    shape = (n_users, n_products)
)

user_sizes = np.array(matrix.sum(axis=1), dtype=np.float32).flatten()

print(f"Matriz esparsa: {n_users:,} usuários × {n_products:,} produtos")
print(f"Não-zeros: {matrix.nnz:,} | Densidade: {matrix.nnz / (n_users * n_products):.4%}")

Matriz esparsa: 131,209 usuários × 49,468 produtos
Não-zeros: 8,474,661 | Densidade: 0.1306%


In [16]:
N_NEIGHBORS = 30
MIN_SHARED = 3
BATCH_SIZE = 500

CACHED_DIR.mkdir(parents = True, exist_ok = True)

if JACCARD_CACHE_PATH.exists():
    print("[CACHE] Parquet encontrado! Carregando dados direto do disco...")
    jaccard_df = pd.read_parquet(JACCARD_CACHE_PATH)
    
    jaccard_neighbors_dict = defaultdict(list)
    
    for user, neighbor, score in zip(jaccard_df['user_id'], jaccard_df['neighbor_id'], jaccard_df['jaccard_score']):
        jaccard_neighbors_dict[user].append((neighbor, score))
        
    jaccard_neighbors = dict(jaccard_neighbors_dict)
    print(f"Dicionário pronto via cache.")
    
else:

    jaccard_neighbors = {}

    for start in range(0, n_users, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_users)

        intersections = (matrix[start:end] @ matrix.T).toarray().astype(np.float32)

        batch_sizes = user_sizes[start:end, np.newaxis]
        neighbor_sizes = user_sizes[np.newaxis, :]
        union = batch_sizes + neighbor_sizes - intersections

        jaccard = np.where(
            (union > 0) & (intersections >= MIN_SHARED),
            intersections / union,
            0.0,
        ).astype(np.float32)

        # Zerar auto-similaridade
        local_idx = np.arange(end - start)
        jaccard[local_idx, start + local_idx] = 0.0

        for i, global_idx in enumerate(range(start, end)):
            user_id = eligible_list[global_idx]
            scores = jaccard[i]
            top_idx = np.argsort(-scores)[:N_NEIGHBORS]
            top_idx = top_idx[scores[top_idx] > 0]
            jaccard_neighbors[user_id] = [
                (eligible_list[j], float(scores[j])) for j in top_idx
            ]

        if start % 10_000 == 0:
            print(f"  {start:,}/{n_users:,} usuários processados...")
        
    print("Convertendo para DataFrame para salvar o arquivo de cache Parquet...")
    users, neighbors, scores = [], [], []
    for u, n_list in jaccard_neighbors.items():
        for n, s in n_list:
            users.append(u)
            neighbors.append(n)
            scores.append(s)
            
    jaccard_df = pd.DataFrame({
        'user_id': users,
        'neighbor_id': neighbors,
        'jaccard_score': scores
    })
    
    jaccard_df.to_parquet(JACCARD_CACHE_PATH, index=False)
    print("Arquivo Parquet persistido com sucesso!")

[CACHE] Parquet encontrado! Carregando dados direto do disco...
Dicionário pronto via cache.


In [17]:
neighbor_counts = pd.Series(
    {uid: len(neighbors) for uid, neighbors in jaccard_neighbors.items()}
)

no_neighbors = (neighbor_counts == 0).sum()

print(f"Usuários sem vizinhos (MIN_SHARED={MIN_SHARED}): {no_neighbors:,} ({100*no_neighbors/n_users:.1f}%)")
print(f"→ Esses usuários receberão fallback para categorias\n")
print("Distribuição de vizinhos por usuário:")
print(neighbor_counts.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).to_string())

Usuários sem vizinhos (MIN_SHARED=3): 0 (0.0%)
→ Esses usuários receberão fallback para categorias

Distribuição de vizinhos por usuário:
count    130203.000000
mean         29.711412
std           2.532263
min           1.000000
10%          30.000000
25%          30.000000
50%          30.000000
75%          30.000000
90%          30.000000
max          30.000000


### 2.3.6 Geração de candidatos v2 — alocação, deduplicação e overflow

In [18]:
CATEGORY_TOP_AISLES = 10
CATEGORY_AISLE_DEPTH = 15  # máx 150 candidatos de categoria por usuário

# Produtos por usuário (necessário para Jaccard pool e deduplicação)
user_products_dict = (
    user_product_freq
    .groupby('user_id')['product_id']
    .apply(set)
    .to_dict()
)

# Recompra por usuário: lista ordenada por rank (buy_count desc)
user_recompra_map = (
    user_product_freq
    .sort_values(['user_id', 'rank'])
    .groupby('user_id')['product_id']
    .apply(list)
    .to_dict()
)

# Categoria por usuário: top produtos dos aisles mais frequentes
top_user_aisles = (
    user_aisle_freq
    .sort_values(['user_id', 'user_aisle_weight'], ascending=[True, False])
    .groupby('user_id')
    .head(CATEGORY_TOP_AISLES)
    [['user_id', 'aisle_id', 'user_aisle_weight']]
)

aisle_top = aisle_product_popularity[
    aisle_product_popularity['aisle_product_rank'] <= CATEGORY_AISLE_DEPTH
][['aisle_id', 'product_id', 'aisle_product_rank']]

category_raw = top_user_aisles.merge(
    aisle_top, 
    on='aisle_id',
    how = 'inner'
)
category_raw['category_score'] = (
    category_raw['user_aisle_weight'] / category_raw['aisle_product_rank']
)
category_raw['category_rank'] = (
    category_raw.groupby('user_id')['category_score']
    .rank(method='first', ascending=False)
    .astype(int)
)

category_pool_dict = (
    category_raw
    .sort_values(['user_id', 'category_rank'])
    .groupby('user_id')['product_id']
    .apply(list)
    .to_dict()
)

print(f"user_products_dict:   {len(user_products_dict):,} usuários")
print(f"user_recompra_map:    {len(user_recompra_map):,} usuários")
print(f"category_pool_dict:   {len(category_pool_dict):,} usuários | média {sum(len(v) for v in category_pool_dict.values()) / len(category_pool_dict):.0f} produtos/usuário")

user_products_dict:   131,209 usuários
user_recompra_map:    131,209 usuários
category_pool_dict:   131,209 usuários | média 143 produtos/usuário


In [19]:
# Produtos dos vizinhos Jaccard não comprados pelo próprio usuário.
# Score = soma dos Jaccard scores dos vizinhos que compraram o produto.
# Produtos com mais vizinhos similares comprando ficam no topo.

jaccard_pool_dict = {}

for user_id in eligible_users:
    own = user_products_dict[user_id] # Evitar produtos de recompra em similaridade
    scores = {}

    for neighbor_id, sim_score in jaccard_neighbors.get(user_id, []):
        for pid in user_products_dict.get(neighbor_id, set()):
            if pid not in own:
                scores[pid] = scores.get(pid, 0.0) + sim_score

    jaccard_pool_dict[user_id] = sorted(scores, key=scores.__getitem__, reverse=True)

no_jaccard = sum(1 for v in jaccard_pool_dict.values() if len(v) == 0)
avg_jaccard = sum(len(v) for v in jaccard_pool_dict.values()) / len(eligible_users)

print(f"Usuários com pool Jaccard vazio: {no_jaccard:,} ({100*no_jaccard/len(eligible_users):.1f}%)")
print(f"Média de produtos Jaccard por usuário: {avg_jaccard:.0f}")

Usuários com pool Jaccard vazio: 1,006 (0.8%)
Média de produtos Jaccard por usuário: 999


In [20]:
def fill_from_pool(pool, selected_set, n_needed):
    """Adiciona até n_needed produtos de pool que não estejam em selected_set."""
    added = []
    for pid in pool:
        if n_needed <= 0:
            break
        if pid not in selected_set:
            added.append(pid)
            n_needed -= 1
    return added


GROUP_CONFIG = {
    'P0-P50':  {'recompra': 50, 'similarity': 200, 'category': 50, 'global': 50},
    'P50-P90': {'recompra': 125, 'similarity': 200, 'category': 25, 'global': 0},
    'P90+':    {'recompra': 175, 'similarity': 150, 'category': 0,  'global': 0},
}

CAP = 350
global_pool_list = global_popularity['product_id'].tolist()
user_group_map = user_group.set_index('user_id')['group'].astype(str).to_dict()

all_candidates = []

for user_id in eligible_users:
    group = user_group_map[user_id]
    cfg = GROUP_CONFIG[group]
    selected = []
    sel_set = set()

    # 1. Recompra
    recompra = fill_from_pool(
        user_recompra_map.get(user_id, [])[:cfg['recompra']], sel_set, cfg['recompra']
    )
    selected.extend(recompra)
    sel_set.update(recompra)
    shortfall = cfg['recompra'] - len(recompra)

    # 2. Similaridade Jaccard (+ shortfall de recompra)
    sim_added = fill_from_pool(
        jaccard_pool_dict.get(user_id, []), sel_set, cfg['similarity'] + shortfall
    )
    selected.extend(sim_added)
    sel_set.update(sim_added)
    shortfall = (cfg['similarity'] + shortfall) - len(sim_added)

    # 3. Categoria (+ shortfall de similaridade)
    cat_added = fill_from_pool(
        category_pool_dict.get(user_id, []), sel_set, cfg['category'] + shortfall
    )
    selected.extend(cat_added)
    sel_set.update(cat_added)
    shortfall = (cfg['category'] + shortfall) - len(cat_added)

    # 4. Global (+ shortfall de categoria — último recurso para todos os grupos)
    global_added = fill_from_pool(
        global_pool_list, sel_set, cfg['global'] + shortfall
    )
    selected.extend(global_added)
    sel_set.update(global_added)

    for pid in selected[:CAP]:
        all_candidates.append((user_id, pid))

candidates_v2 = pd.DataFrame(all_candidates, columns=['user_id', 'product_id'])

assert not candidates_v2.duplicated(subset=['user_id', 'product_id']).any(), "Pares duplicados."
assert candidates_v2['user_id'].isin(eligible_users).all(), "Usuários inelegíveis."

print(f"Candidatos v2: {len(candidates_v2):,} pares")
print(f"Usuários cobertos: {candidates_v2['user_id'].nunique():,}")
print(f"Média de candidatos/usuário: {len(candidates_v2) / candidates_v2['user_id'].nunique():.1f}")
candidates_v2.head()

Candidatos v2: 45,519,000 pares
Usuários cobertos: 131,209
Média de candidatos/usuário: 346.9


,user_id,product_id
0,1,196
1,1,12427
2,1,10258
3,1,25133
4,1,13032


In [21]:
cands_per_user_v2 = (
    candidates_v2.groupby('user_id')['product_id']
    .count()
    .rename('candidate_count')
)

below_cap = (cands_per_user_v2 < CAP).sum()
print(f"Usuários com menos de {CAP} candidatos: {below_cap:,} ({100*below_cap/len(eligible_users):.1f}%)\n")

print("Distribuição de candidatos por usuário:")
print(cands_per_user_v2.describe().to_string())

print("\nMédia de candidatos por grupo:")
group_counts = (
    candidates_v2
    .merge(user_group[['user_id', 'group']], on='user_id', how = 'inner')
    .groupby('group', observed = False)
    .size()
)
users_per_group = user_group['group'].value_counts().sort_index()
print((group_counts / users_per_group).round(1).rename('avg_candidates').to_frame().to_string())

Usuários com menos de 350 candidatos: 14,218 (10.8%)

Distribuição de candidatos por usuário:
count    131209.000000
mean        346.919800
std           9.937242
min         250.000000
25%         350.000000
50%         350.000000
75%         350.000000
max         350.000000

Média de candidatos por grupo:
         avg_candidates
group                  
P0-P50            348.8
P50-P90           350.0
P90+              325.0


### 2.3.7 Recall ceiling combinado — candidatos v2

In [22]:
covered_v2 = candidates_v2.merge(
    train_set, on=['user_id', 'product_id'], how='inner'
)
covered_v2_group = covered_v2.merge(
    user_group[['user_id', 'group']], on='user_id', how = 'inner'
)

total_train = len(train_set)
recall_v2 = len(covered_v2) / total_train

RECALL_V1_BASELINE = 0.5986

print(f"Produtos no train (total):           {total_train:,}")
print(f"Cobertos pelos candidatos v2:        {len(covered_v2):,}")
print(f"\nRecall ceiling v2 (combinado):       {recall_v2:.2%}")
print(f"Recall ceiling v1 (baseline):        {RECALL_V1_BASELINE:.2%}")
print(f"Ganho:                               +{(recall_v2 - RECALL_V1_BASELINE)*100:.2f} pp")
print(f"\nCritério de aceite (> 59.86%):       {'✓ PASSOU' if recall_v2 > RECALL_V1_BASELINE else '✗ FALHOU — revisar alocação'}")

print("\nRecall ceiling v2 por grupo (%):")
for grp in ['P0-P50', 'P50-P90', 'P90+']:
    total_grp = (train_with_group['group'] == grp).sum()
    hits_grp = (covered_v2_group['group'] == grp).sum()
    print(f"  {grp}: {100 * hits_grp / total_grp:.2f}%")

Produtos no train (total):           1,384,617
Cobertos pelos candidatos v2:        991,122

Recall ceiling v2 (combinado):       71.58%
Recall ceiling v1 (baseline):        59.86%
Ganho:                               +11.72 pp

Critério de aceite (> 59.86%):       ✓ PASSOU

Recall ceiling v2 por grupo (%):
  P0-P50: 65.98%
  P50-P90: 73.91%
  P90+: 77.77%


### 2.4 Construção do target supervisionado

In [23]:
modeling_df = candidates_v2.merge(
    train_products,
    on=['user_id', 'product_id'],
    how='left'
)

modeling_df['target'] = modeling_df['target'].fillna(0).astype(int)

assert len(modeling_df) == len(candidates_v2), (
    "O merge com train_products alterou a quantidade de candidatos."
)
assert modeling_df['target'].isin([0, 1]).all(), (
    "A coluna target contém valores fora do esperado {0, 1}."
)
assert not modeling_df.duplicated(subset=['user_id', 'product_id']).any(), (
    "Existem pares user_id-product_id duplicados no dataset de modelagem."
)

print(f"Dataset de modelagem: {len(modeling_df):,} pares")
print(f"Positivos (target=1): {(modeling_df['target'] == 1).sum():,}")
print(f"Negativos (target=0): {(modeling_df['target'] == 0).sum():,}")
print(f"Taxa de positivos:    {(modeling_df['target'] == 1).mean():.4%}")

Dataset de modelagem: 45,519,000 pares
Positivos (target=1): 991,122
Negativos (target=0): 44,527,878
Taxa de positivos:    2.1774%


### 2.4.1 Diagnóstico do target supervisionado

In [24]:
target_dist = (
    modeling_df['target']
    .value_counts()
    .reset_index(name='count')
    .assign(pct=lambda d: 100 * d['count'] / d['count'].sum())
    .sort_values('target')
)

target_dist

,target,count,pct
0,0,44527878,97.822619
1,1,991122,2.177381


In [25]:
pos_by_group = (
    modeling_df
    .merge(user_group[['user_id', 'group']], on='user_id')
    .groupby('group', observed=False)['target']
    .agg(total='count', positivos='sum')
    .reset_index()
    .assign(taxa_positivos=lambda d: 100 * d['positivos'] / d['total'])
)

print(pos_by_group.to_string())

     group     total  positivos  taxa_positivos
0   P0-P50  23304400     335643        1.440256
1  P50-P90  17966200     491808        2.737407
2     P90+   4248400     163671        3.852533


---

## 3. Persistência dos candidatos supervisionados

Esta seção salva o artefato intermediário produzido pelo notebook: os pares `user_id-product_id` candidatos com o target supervisionado.

Esse arquivo será usado como input do notebook `04-feature-engineering.ipynb`, onde as features históricas serão calculadas e consolidadas.

In [26]:
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

modeling_df.to_parquet(OUTPUT_CANDIDATES_PATH, index=False)

assert OUTPUT_CANDIDATES_PATH.exists(), (
    f"Arquivo de candidatos não foi salvo em: {OUTPUT_CANDIDATES_PATH}"
)

print(f"Candidatos supervisionados salvos em: {OUTPUT_CANDIDATES_PATH}")
print(f"Linhas:   {len(modeling_df):,}")
print(f"Usuários: {modeling_df['user_id'].nunique():,}")
print(f"Produtos: {modeling_df['product_id'].nunique():,}")

Candidatos supervisionados salvos em: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/candidates_v2_with_target.parquet
Linhas:   45,519,000
Usuários: 131,209
Produtos: 49,386
